# Part 2: Streaming application using Spark Structured Streaming  
In this task, you will implement Spark Structured Streaming to consume the data from task 1 and perform a prediction.    
Important: 
- This task uses PySpark Structured Streaming with PySpark Dataframe APIs and PySpark ML.
- You also need your pipeline model from A2A to make predictions and persist the results.
- Note for the prediction related to event time: in a real scenario, you should use accident_ts as the event time; however, since we are simulating streaming and your model was trained with the time column as a feature, you can choose to use the time column or accident_ts.


1. Write code to create a SparkSession, which 1) uses four cores with a proper application name; 2) uses the UK/London timezone; 3) ensures a checkpoint location has been set.

In [1]:
from pyspark.sql import SparkSession
from pyspark import SparkConf

# Stop existing Spark session if one is already running
try:
    spark.stop()
except:
    pass

conf = SparkConf() \
    .setAppName("FIT5202_A2B_Road_Accident_Streaming_Prediction") \
    .setMaster("local[4]") \
    .set("spark.sql.session.timeZone", "Europe/London") \
    .set("spark.sql.streaming.checkpointLocation", "A2B/checkpoint") \
    .set("spark.sql.files.maxPartitionBytes", "32m") \
    .set("spark.sql.shuffle.partitions", "4") \
    .set("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1")

spark = SparkSession.builder \
    .config(conf=conf) \
    .getOrCreate()

sc = spark.sparkContext

print("Spark application name:", sc.appName)
print("Spark master:", sc.master)
print("Spark version:", spark.version)
print("Session timezone:", spark.conf.get("spark.sql.session.timeZone"))
print("Checkpoint location:", spark.conf.get("spark.sql.streaming.checkpointLocation"))
print("Kafka package:", spark.conf.get("spark.jars.packages"))

Spark application name: FIT5202_A2B_Road_Accident_Streaming_Prediction
Spark master: local[4]
Spark version: 4.1.1
Session timezone: Europe/London
Checkpoint location: A2B/checkpoint
Kafka package: org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1


2. Write code to define the data schema for the data files. Load the static datasets into data frames. (You can reuse your code from 2A.) In a car accident, we collection streaming information like the realtime road condition, but the vehicle information is static and can be read from the vehicle registration database.

In [2]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)

# Schema for streaming_collision.csv
# These are the actual columns printed from the CSV file.
# accident_ts is not in the original CSV, but it is added by the Kafka producer.
# Therefore, we include accident_ts because Spark Streaming will receive it.

streaming_collision_schema = StructType([
    StructField("collision_index", StringType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("date", StringType(), True),
    StructField("time", StringType(), True),
    StructField("road_type", IntegerType(), True),
    StructField("speed_limit", IntegerType(), True),
    StructField("junction_detail", IntegerType(), True),
    StructField("junction_control", IntegerType(), True),
    StructField("pedestrian_crossing", IntegerType(), True),
    StructField("light_conditions", IntegerType(), True),
    StructField("weather_conditions", IntegerType(), True),
    StructField("road_surface_conditions", IntegerType(), True),
    StructField("carriageway_hazards", IntegerType(), True),
    StructField("urban_or_rural_area", IntegerType(), True),
    StructField("area", StringType(), True),
    StructField("accident_ts", StringType(), True)
])

# Schema for vehicle.csv
# These are the actual columns printed from the vehicle CSV file.
# This is the static dataset used like a vehicle registration database.

vehicle_schema = StructType([
    StructField("collision_index", StringType(), True),
    StructField("vehicle_reference", IntegerType(), True),
    StructField("vehicle_type", IntegerType(), True),
    StructField("vehicle_manoeuvre", IntegerType(), True),
    StructField("junction_location", IntegerType(), True),
    StructField("skidding_and_overturning", IntegerType(), True),
    StructField("hit_object_in_carriageway", IntegerType(), True),
    StructField("first_point_of_impact", IntegerType(), True),
    StructField("sex_of_driver", IntegerType(), True),
    StructField("age_of_driver", IntegerType(), True),
    StructField("engine_capacity_cc", IntegerType(), True),
    StructField("propulsion_code", IntegerType(), True),
    StructField("age_of_vehicle", IntegerType(), True)
])

# Load static vehicle dataset
base_path = "A2B/"

vehicle_df = spark.read.csv(
    base_path + "vehicle.csv",
    header=True,
    schema=vehicle_schema
)

# Check the loaded static dataset
print("Schema of vehicle_df:")
vehicle_df.printSchema()

print("Number of rows in vehicle_df:", vehicle_df.count())

print("First 5 rows of vehicle_df:")
vehicle_df.show(5, truncate=False)

Schema of vehicle_df:
root
 |-- collision_index: string (nullable = true)
 |-- vehicle_reference: integer (nullable = true)
 |-- vehicle_type: integer (nullable = true)
 |-- vehicle_manoeuvre: integer (nullable = true)
 |-- junction_location: integer (nullable = true)
 |-- skidding_and_overturning: integer (nullable = true)
 |-- hit_object_in_carriageway: integer (nullable = true)
 |-- first_point_of_impact: integer (nullable = true)
 |-- sex_of_driver: integer (nullable = true)
 |-- age_of_driver: integer (nullable = true)
 |-- engine_capacity_cc: integer (nullable = true)
 |-- propulsion_code: integer (nullable = true)
 |-- age_of_vehicle: integer (nullable = true)

Number of rows in vehicle_df: 61081
First 5 rows of vehicle_df:
+---------------+-----------------+------------+-----------------+-----------------+------------------------+-------------------------+---------------------+-------------+-------------+------------------+---------------+--------------+
|collision_index|vehicl

3. Using the Kafka topic from the producer in Task 1, ingest the streaming data into Spark Streaming, assuming all data comes in the String format. Except for the 'accident_ts' column, you shall receive it as a numeric type. Then, the data frames should be transformed into the appropriate types.

In [3]:
from pyspark.sql.functions import (
    col, from_json, explode, to_timestamp,
    from_unixtime, when
)
from pyspark.sql.types import (
    StructType, StructField,
    StringType, ArrayType, LongType
)


kafka_topic = "accident_stream"
hostip = "kafka"

# Raw schema for Kafka JSON message
# In the producer, CSV values are sent as strings.
# accident_ts could be a timestamp string or numeric timestamp depending on producer format.
# Therefore, we first read accident_ts as string, then convert it later.

raw_collision_schema = StructType([
    StructField("collision_index", StringType(), True),
    StructField("longitude", StringType(), True),
    StructField("latitude", StringType(), True),
    StructField("date", StringType(), True),
    StructField("time", StringType(), True),
    StructField("road_type", StringType(), True),
    StructField("speed_limit", StringType(), True),
    StructField("junction_detail", StringType(), True),
    StructField("junction_control", StringType(), True),
    StructField("pedestrian_crossing", StringType(), True),
    StructField("light_conditions", StringType(), True),
    StructField("weather_conditions", StringType(), True),
    StructField("road_surface_conditions", StringType(), True),
    StructField("carriageway_hazards", StringType(), True),
    StructField("urban_or_rural_area", StringType(), True),
    StructField("area", StringType(), True),
    StructField("accident_ts", LongType(), True)
])

# Since each Kafka message is a batch/list of accident records,
# the JSON value should be parsed as an array of structs.
raw_collision_array_schema = ArrayType(raw_collision_schema)

# Read streaming data from Kafka
kafka_stream_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", f"{hostip}:9092") \
    .option("subscribe", kafka_topic) \
    .option("startingOffsets", "latest") \
    .load()

# Deserialize Kafka value and explode the JSON array into individual rows
parsed_stream_df = kafka_stream_df.select(
    col("key").cast("string").alias("kafka_key"),
    col("value").cast("string").alias("json_value"),
    col("timestamp").alias("kafka_timestamp")
)

collision_stream_raw_df = parsed_stream_df.select(
    "kafka_key",
    "kafka_timestamp",
    explode(from_json(col("json_value"), raw_collision_array_schema)).alias("data")
).select(
    "kafka_key",
    "kafka_timestamp",
    "data.*"
)

# Convert columns to appropriate data types
collision_stream_df = collision_stream_raw_df \
    .withColumn("longitude", col("longitude").cast("double")) \
    .withColumn("latitude", col("latitude").cast("double")) \
    .withColumn("road_type", col("road_type").cast("int")) \
    .withColumn("speed_limit", col("speed_limit").cast("int")) \
    .withColumn("junction_detail", col("junction_detail").cast("int")) \
    .withColumn("junction_control", col("junction_control").cast("int")) \
    .withColumn("pedestrian_crossing", col("pedestrian_crossing").cast("int")) \
    .withColumn("light_conditions", col("light_conditions").cast("int")) \
    .withColumn("weather_conditions", col("weather_conditions").cast("int")) \
    .withColumn("road_surface_conditions", col("road_surface_conditions").cast("int")) \
    .withColumn("carriageway_hazards", col("carriageway_hazards").cast("int")) \
    .withColumn("urban_or_rural_area", col("urban_or_rural_area").cast("int")) \
    .withColumn("accident_event_ts", to_timestamp(from_unixtime(col("accident_ts"))))

# Check schema after type conversion
collision_stream_df.printSchema()

root
 |-- kafka_key: string (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- collision_index: string (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- date: string (nullable = true)
 |-- time: string (nullable = true)
 |-- road_type: integer (nullable = true)
 |-- speed_limit: integer (nullable = true)
 |-- junction_detail: integer (nullable = true)
 |-- junction_control: integer (nullable = true)
 |-- pedestrian_crossing: integer (nullable = true)
 |-- light_conditions: integer (nullable = true)
 |-- weather_conditions: integer (nullable = true)
 |-- road_surface_conditions: integer (nullable = true)
 |-- carriageway_hazards: integer (nullable = true)
 |-- urban_or_rural_area: integer (nullable = true)
 |-- area: string (nullable = true)
 |-- accident_ts: long (nullable = true)
 |-- accident_event_ts: timestamp (nullable = true)



### Discussion

The Kafka stream from the `accident_stream` topic was successfully ingested into Spark Streaming. Since each Kafka message contains a JSON array of accident records, the data was parsed and exploded into individual rows. The schema output confirms that the main columns were converted into suitable types, including `double` for coordinates, `integer` for road/environment fields, `long` for `accident_ts`, and `timestamp` for `accident_event_ts`. This prepared the streaming data for later watermarking, window aggregation, and prediction tasks.


4. Use a watermark on accident_ts. If data points are received 30 seconds late, discard the data. (note: in a local environment like your laptop, late arrival or delayed processing may never happen.)

In [4]:
# The original accident_ts column is received from Kafka.
# In Task 2.3, it was converted to accident_event_ts as TimestampType.
# Spark watermark requires a timestamp column, so we apply the watermark
# to accident_event_ts.
# Records more than 30 seconds late can be discarded by Spark
# when stateful streaming operations are performed later.

collision_stream_watermark_df = collision_stream_df \
    .filter(col("accident_event_ts").isNotNull()) \
    .withWatermark("accident_event_ts", "30 seconds")

# Check schema after applying watermark
collision_stream_watermark_df.printSchema()

root
 |-- kafka_key: string (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- collision_index: string (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- date: string (nullable = true)
 |-- time: string (nullable = true)
 |-- road_type: integer (nullable = true)
 |-- speed_limit: integer (nullable = true)
 |-- junction_detail: integer (nullable = true)
 |-- junction_control: integer (nullable = true)
 |-- pedestrian_crossing: integer (nullable = true)
 |-- light_conditions: integer (nullable = true)
 |-- weather_conditions: integer (nullable = true)
 |-- road_surface_conditions: integer (nullable = true)
 |-- carriageway_hazards: integer (nullable = true)
 |-- urban_or_rural_area: integer (nullable = true)
 |-- area: string (nullable = true)
 |-- accident_ts: long (nullable = true)
 |-- accident_event_ts: timestamp (nullable = true)



### Discussion 

A 30-second watermark was applied to the converted event-time column `accident_event_ts`, which was derived from the producer-added `accident_ts` field. The schema confirms that `accident_event_ts` is stored as a timestamp, allowing Spark to perform event-time streaming operations. This watermark enables Spark to discard records that arrive more than 30 seconds late during later window-based aggregations.

5. Perform the necessary transformation you used in A2A. (note: every student may have used different features, feel free to reuse the code you have written in A2A. If you built an end-to-end pipeline, you can ignore this task.) 

In [5]:
# In A2A, the saved model was trained on a prepared model_df.
# Therefore, the streaming data must be transformed into the same
# input feature structure before applying the saved PipelineModel.

from pyspark.sql.functions import (
    col, count, avg, when, round as spark_round, split
)
from pyspark.sql.types import DoubleType, StringType


# Aggregate static vehicle data to collision level
# This follows the same idea used in A2A:
# vehicle.csv may contain multiple vehicles per collision, so we aggregate
# vehicle-level attributes into collision-level features.

vehicle_agg_df = vehicle_df.groupBy("collision_index").agg(
    count("*").alias("vehicle_count"),

    spark_round(
        avg(when(col("age_of_driver") > 0, col("age_of_driver"))),
        2
    ).alias("avg_driver_age"),

    spark_round(
        avg(when(col("engine_capacity_cc") > 0, col("engine_capacity_cc"))),
        2
    ).alias("avg_engine_capacity_cc")
)

print("Schema of aggregated static vehicle data:")
vehicle_agg_df.printSchema()

print("Sample aggregated vehicle data:")
vehicle_agg_df.show(5, truncate=False)


# Join streaming collision data with static vehicle features
# collision_stream_watermark_df is the streaming DataFrame created in
# Task 2.4. vehicle_agg_df is static, so this is a stream-static join.

stream_joined_df = collision_stream_watermark_df.join(
    vehicle_agg_df,
    on="collision_index",
    how="left"
)


# Fill missing vehicle aggregate values using the same logic as A2A
# In A2A, median imputation learned:
# avg_driver_age = 34.5
# avg_engine_capacity_cc = 1592.5
# If a streaming collision has no matching vehicle record, vehicle_count is
# filled as 0 and the two numeric vehicle features are filled with the A2A median values

stream_imputed_df = stream_joined_df \
    .withColumn(
        "vehicle_count",
        when(col("vehicle_count").isNull(), 0).otherwise(col("vehicle_count"))
    ) \
    .withColumn(
        "avg_driver_age",
        when(col("avg_driver_age").isNull(), 34.5).otherwise(col("avg_driver_age"))
    ) \
    .withColumn(
        "avg_engine_capacity_cc",
        when(col("avg_engine_capacity_cc").isNull(), 1592.5)
        .otherwise(col("avg_engine_capacity_cc"))
    )


# Create engineered features used in A2A
# A2A created:
# hour from time
# Peak_Traffic: Peak for 07:00-09:00 and 16:00-18:00
# High_Risk_Environment based on light, weather, and road surface codes

stream_feature_df = stream_imputed_df \
    .withColumn(
        "hour",
        split(col("time"), ":").getItem(0).cast("int")
    ) \
    .withColumn(
        "Peak_Traffic",
        when(
            ((col("hour") >= 7) & (col("hour") <= 9)) |
            ((col("hour") >= 16) & (col("hour") <= 18)),
            "Peak"
        ).otherwise("Off_Peak")
    ) \
    .withColumn(
        "High_Risk_Environment",
        when(
            (col("light_conditions").isin([4, 5, 6, 7])) |
            (col("weather_conditions").isin([2, 3, 4, 5, 6, 7])) |
            (col("road_surface_conditions").isin([2, 3, 4, 5, 6, 7])),
            "High_Risk"
        ).otherwise("Normal_Risk")
    )


# Prepare final model input columns in the same types as A2A model_df
# Numeric features are cast to DoubleType.
# Integer-coded categorical features are cast to StringType.
# This matches the A2A pipeline input before StringIndexer/OneHotEncoder

numeric_feature_cols = [
    "speed_limit",
    "vehicle_count",
    "avg_driver_age",
    "avg_engine_capacity_cc",
    "hour"
]

categorical_feature_cols = [
    "road_type",
    "junction_detail",
    "junction_control",
    "pedestrian_crossing",
    "light_conditions",
    "weather_conditions",
    "road_surface_conditions",
    "carriageway_hazards",
    "urban_or_rural_area",
    "area",
    "Peak_Traffic",
    "High_Risk_Environment"
]

# Cast numeric columns to double
stream_model_input_df = stream_feature_df
for c in numeric_feature_cols:
    stream_model_input_df = stream_model_input_df.withColumn(
        c,
        col(c).cast(DoubleType())
    )

# Cast categorical columns to string
for c in categorical_feature_cols:
    stream_model_input_df = stream_model_input_df.withColumn(
        c,
        col(c).cast(StringType())
    )

# Keep useful ID/time columns plus the model input columns
stream_model_input_df = stream_model_input_df.select(
    "collision_index",
    "accident_ts",
    "accident_event_ts",
    "kafka_timestamp",
    *numeric_feature_cols,
    *categorical_feature_cols
)

# Check final streaming model input schema
stream_model_input_df.printSchema()

Schema of aggregated static vehicle data:
root
 |-- collision_index: string (nullable = true)
 |-- vehicle_count: long (nullable = false)
 |-- avg_driver_age: double (nullable = true)
 |-- avg_engine_capacity_cc: double (nullable = true)

Sample aggregated vehicle data:
+---------------+-------------+--------------+----------------------+
|collision_index|vehicle_count|avg_driver_age|avg_engine_capacity_cc|
+---------------+-------------+--------------+----------------------+
|2025000000004  |2            |29.0          |2069.0                |
|2025000000006  |1            |19.0          |NULL                  |
|2025000000012  |4            |68.5          |1123.5                |
|2025000000017  |2            |46.5          |1046.0                |
|2025000000026  |1            |47.0          |1686.0                |
+---------------+-------------+--------------+----------------------+
only showing top 5 rows
root
 |-- collision_index: string (nullable = true)
 |-- accident_ts: long 

### Discussion

+ The streaming collision data was transformed into the same feature structure used by the saved A2A pipeline model. Static vehicle data was first aggregated at the `collision_index` level to create collision-level features such as `vehicle_count`, `avg_driver_age`, and `avg_engine_capacity_cc`. These features were then joined with the streaming collision data.

+ Missing vehicle aggregate values were filled using the same imputation values from A2A, and engineered features such as `hour`, `Peak_Traffic`, and `High_Risk_Environment` were recreated. The final schema confirms that numeric features were cast to `double` and categorical features were cast to `string`, matching the expected input format of the A2A model pipeline.


6. Load your pipeline model and perform the following aggregations:  
a) Make predictions and print high-severity accidents (>7) every 5 seconds.   
b) Every 10 seconds, print the total number of accidents for each severity.  
c) Every 30 seconds, for each local district with data, print the total number of low (1-3), medium (4-6) and high severity (7-10) accidents.  

In [6]:
import shutil
import os

# List of checkpoint folders used by previous versions of Task 2.6
# These are mainly for memory-sink display queries in 6a, 6b and 6c
memory_checkpoint_paths = [
    "A2B/checkpoint/high_severity_table",
    "A2B/checkpoint/high_severity_table_v2",
    "A2B/checkpoint/severity_count_table",
    "A2B/checkpoint/severity_count_table_v2",
    "A2B/checkpoint/district_severity_table",
    "A2B/checkpoint/district_severity_table_v2"
]

# Delete each checkpoint folder if it already exists
# This makes the next streaming run start cleanly
for path in memory_checkpoint_paths:
    if os.path.exists(path):
        shutil.rmtree(path)
        print("Deleted:", path)

print("Memory checkpoint clean-up completed.")

Deleted: A2B/checkpoint/high_severity_table_v2
Deleted: A2B/checkpoint/severity_count_table
Deleted: A2B/checkpoint/district_severity_table
Memory checkpoint clean-up completed.


In [7]:
# 6a
from pyspark.ml import PipelineModel
from pyspark.sql.functions import col, round as spark_round, least, greatest, lit

# Load saved A2A model
# Try the first path. If it fails, use the second path

model_path = "../Assignment2_partA/A2A_best_severity_model"

try:
    severity_model = PipelineModel.load(model_path)
    print("Model loaded from:", model_path)
except Exception as e:
    print("First model path failed:", model_path)
    print("Trying alternative path...")

    model_path = "A2A_best_severity_model"
    severity_model = PipelineModel.load(model_path)
    print("Model loaded from:", model_path)


# Make predictions using the saved A2A PipelineModel
prediction_stream_df = severity_model.transform(stream_model_input_df)

# Convert continuous regression prediction into severity rating 1-10
# This makes the output easier to interpret for dashboard/aggregation.
prediction_output_df = prediction_stream_df \
    .withColumn("predicted_severity_raw", col("prediction")) \
    .withColumn("predicted_severity", spark_round(col("prediction")).cast("int")) \
    .withColumn("predicted_severity", greatest(lit(1), least(lit(10), col("predicted_severity"))))

# High severity accidents are predicted severity > 7
high_severity_df = prediction_output_df.filter(
    col("predicted_severity") > 7
).select(
    "collision_index",
    "accident_event_ts",
    "area",
    "speed_limit",
    "vehicle_count",
    "Peak_Traffic",
    "High_Risk_Environment",
    "predicted_severity_raw",
    "predicted_severity"
)

# Print high-severity accidents every 5 seconds
query_6a = high_severity_df.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", "false") \
    .option("numRows", 20) \
    .option("checkpointLocation", "A2B/checkpoint/query_6a_high_severity_v3") \
    .trigger(processingTime="5 seconds") \
    .start()

print("Query 6a started: printing high-severity accidents every 5 seconds.")

Model loaded from: ../Assignment2_partA/A2A_best_severity_model
Query 6a started: printing high-severity accidents every 5 seconds.


In [8]:
# Display Check: Write high-severity accidents to memory sink
# Use a new table name to avoid old memory-sink checkpoint recovery errors

query_6a_memory = high_severity_df.writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("high_severity_table_v2") \
    .trigger(processingTime="5 seconds") \
    .start()

print("6a memory query started.")

6a memory query started.


In [9]:
spark.sql("""
    SELECT *
    FROM high_severity_table_v2
    ORDER BY accident_event_ts
""").show(20, truncate=False)

+---------------+-----------------+----+-----------+-------------+------------+---------------------+----------------------+------------------+
|collision_index|accident_event_ts|area|speed_limit|vehicle_count|Peak_Traffic|High_Risk_Environment|predicted_severity_raw|predicted_severity|
+---------------+-----------------+----+-----------+-------------+------------+---------------------+----------------------+------------------+
+---------------+-----------------+----+-----------+-------------+------------+---------------------+----------------------+------------------+



### Discussion

The saved A2A pipeline model was successfully loaded from `../Assignment2_partA/A2A_best_severity_model` and applied to the prepared streaming input DataFrame. The continuous prediction output was converted into an integer severity rating from 1 to 10 for easier interpretation. High-severity accidents were then filtered using `predicted_severity > 7` and printed every 5 seconds. In the displayed result, no high-severity records were shown, which indicates that no current streaming records satisfied the `predicted_severity > 7` condition during this run.


In [10]:
# 6b
# Every 10 seconds, print total accidents for each severity

from pyspark.sql.functions import window, count, col

severity_count_df = prediction_output_df \
    .groupBy(
        window(col("accident_event_ts"), "10 seconds"),
        col("predicted_severity")
    ) \
    .agg(
        count("*").alias("total_accidents")
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "predicted_severity",
        "total_accidents"
    )

query_6b = severity_count_df.writeStream \
    .outputMode("complete") \
    .format("console") \
    .option("truncate", "false") \
    .option("numRows", 50) \
    .option("checkpointLocation", "A2B/checkpoint/query_6b_severity_counts_v3") \
    .trigger(processingTime="10 seconds") \
    .start()

print("Query 6b started: printing severity counts every 10 seconds.")

Query 6b started: printing severity counts every 10 seconds.


In [11]:
# Display Check: Write severity counts to memory sink

query_6b_memory = severity_count_df.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("severity_count_table") \
    .trigger(processingTime="10 seconds") \
    .start()

print("6b memory query started.")

6b memory query started.


In [12]:
spark.sql("""
    SELECT *
    FROM severity_count_table
    ORDER BY window_start, predicted_severity
""").show(50, truncate=False)

+-------------------+-------------------+------------------+---------------+
|window_start       |window_end         |predicted_severity|total_accidents|
+-------------------+-------------------+------------------+---------------+
|2026-05-28 04:56:20|2026-05-28 04:56:30|2                 |97             |
|2026-05-28 04:56:20|2026-05-28 04:56:30|3                 |237            |
|2026-05-28 04:56:20|2026-05-28 04:56:30|4                 |271            |
|2026-05-28 04:56:20|2026-05-28 04:56:30|5                 |47             |
|2026-05-28 04:56:20|2026-05-28 04:56:30|6                 |28             |
|2026-05-28 04:56:30|2026-05-28 04:56:40|2                 |94             |
|2026-05-28 04:56:30|2026-05-28 04:56:40|3                 |235            |
|2026-05-28 04:56:30|2026-05-28 04:56:40|4                 |278            |
|2026-05-28 04:56:30|2026-05-28 04:56:40|5                 |43             |
|2026-05-28 04:56:30|2026-05-28 04:56:40|6                 |20             |

### Discussion

The model predictions were aggregated using 10-second tumbling windows to count the total number of accidents for each predicted severity level. The output shows that most predicted accidents fall between severity levels 2 and 6, with severity 3 and 4 having the highest counts in the displayed windows. This satisfies Task 2.6b because the streaming query prints severity-level accident counts every 10 seconds.


In [13]:
# 6c
# Every 30 seconds, print low/medium/high severity counts by area

from pyspark.sql.functions import sum as spark_sum, count, window, when, col

district_severity_df = prediction_output_df \
    .withColumn(
        "severity_group",
        when(col("predicted_severity").between(1, 3), "low")
        .when(col("predicted_severity").between(4, 6), "medium")
        .when(col("predicted_severity").between(7, 10), "high")
        .otherwise("unknown")
    ) \
    .groupBy(
        window(col("accident_event_ts"), "30 seconds"),
        col("area").alias("local_district")
    ) \
    .agg(
        spark_sum(when(col("severity_group") == "low", 1).otherwise(0)).alias("low_severity_count"),
        spark_sum(when(col("severity_group") == "medium", 1).otherwise(0)).alias("medium_severity_count"),
        spark_sum(when(col("severity_group") == "high", 1).otherwise(0)).alias("high_severity_count"),
        count("*").alias("total_accidents")
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "local_district",
        "low_severity_count",
        "medium_severity_count",
        "high_severity_count",
        "total_accidents"
    )

query_6c = district_severity_df.writeStream \
    .outputMode("complete") \
    .format("console") \
    .option("truncate", "false") \
    .option("numRows", 50) \
    .option("checkpointLocation", "A2B/checkpoint/query_6c_district_counts_v3") \
    .trigger(processingTime="30 seconds") \
    .start()

print("Query 6c started: printing district severity summary every 30 seconds.")

Query 6c started: printing district severity summary every 30 seconds.


In [14]:
# Display Check: Write district severity summary to memory sink

query_6c_memory = district_severity_df.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("district_severity_table") \
    .trigger(processingTime="30 seconds") \
    .start()

print("6c memory query started.")

6c memory query started.


In [15]:
spark.sql("""
    SELECT *
    FROM district_severity_table
    ORDER BY window_start, local_district
""").show(50, truncate=False)

+-------------------+-------------------+-------------------+------------------+---------------------+-------------------+---------------+
|window_start       |window_end         |local_district     |low_severity_count|medium_severity_count|high_severity_count|total_accidents|
+-------------------+-------------------+-------------------+------------------+---------------------+-------------------+---------------+
|2026-05-28 04:57:00|2026-05-28 04:57:30|Avon and Somerset  |34                |18                   |0                  |52             |
|2026-05-28 04:57:00|2026-05-28 04:57:30|Bedfordshire       |9                 |16                   |0                  |25             |
|2026-05-28 04:57:00|2026-05-28 04:57:30|Cambridgeshire     |19                |15                   |0                  |34             |
|2026-05-28 04:57:00|2026-05-28 04:57:30|Cheshire           |14                |15                   |0                  |29             |
|2026-05-28 04:57:00|2026-0

### Discussion 

The predicted severity values were grouped into low severity (1–3), medium severity (4–6), and high severity (7–10). The stream was then aggregated using 30-second tumbling windows for each local district. The displayed output shows the number of low, medium, high, and total accidents for each district with data. In the sample output, most districts have low and medium severity counts, while the high severity count is 0 for the displayed rows. This satisfies Task 2.6c because the query prints district-level severity summaries every 30 seconds.


7. Save the data from 6 to Parquet files as streams. (Hint: Parquet files support streaming writing/reading. The file should keep updating while new batches arrive.)

In [16]:
# Stop old active streaming queries 
for q in spark.streams.active:
    q.stop()

print("Active streaming queries:", len(spark.streams.active))

Active streaming queries: 0


In [17]:
# Clean old Parquet output and checkpoint folders
# This avoids checkpoint/path conflicts when rerunning the notebook.

import shutil
import os

task7_paths = [
    "A2B/parquet/high_severity_accidents_v3",
    "A2B/parquet/severity_counts_v3",
    "A2B/parquet/district_severity_summary_v3",
    "A2B/checkpoint/query_7a_high_severity_parquet_v3",
    "A2B/checkpoint/query_7b_severity_counts_parquet_v3",
    "A2B/checkpoint/query_7c_district_severity_parquet_v3"
]

for path in task7_paths:
    if os.path.exists(path):
        shutil.rmtree(path)
        print("Deleted:", path)

print("Task 7 clean-up completed.")

Deleted: A2B/parquet/high_severity_accidents_v3
Deleted: A2B/parquet/severity_counts_v3
Deleted: A2B/parquet/district_severity_summary_v3
Deleted: A2B/checkpoint/query_7a_high_severity_parquet_v3
Deleted: A2B/checkpoint/query_7b_severity_counts_parquet_v3
Deleted: A2B/checkpoint/query_7c_district_severity_parquet_v3
Task 7 clean-up completed.


In [18]:
# 7a(save 6a)
# Save high-severity accidents from 6a to Parquet

query_7a = high_severity_df.writeStream \
    .outputMode("append") \
    .format("parquet") \
    .option("path", "A2B/parquet/high_severity_accidents_v3") \
    .option("checkpointLocation", "A2B/checkpoint/query_7a_high_severity_parquet_v3") \
    .trigger(processingTime="5 seconds") \
    .start()

print("Query 7a started: saving high-severity accidents to Parquet.")

Query 7a started: saving high-severity accidents to Parquet.


In [19]:
# 7b(save 6b)
# Save total accidents for each severity to Parquet

query_7b = severity_count_df.writeStream \
    .outputMode("append") \
    .format("parquet") \
    .option("path", "A2B/parquet/severity_counts_v3") \
    .option("checkpointLocation", "A2B/checkpoint/query_7b_severity_counts_parquet_v3") \
    .trigger(processingTime="10 seconds") \
    .start()

print("Query 7b started: saving severity counts to Parquet.")

Query 7b started: saving severity counts to Parquet.


In [20]:
# 7c(save 6c)
# Save district severity summary to Parquet

query_7c = district_severity_df.writeStream \
    .outputMode("append") \
    .format("parquet") \
    .option("path", "A2B/parquet/district_severity_summary_v3") \
    .option("checkpointLocation", "A2B/checkpoint/query_7c_district_severity_parquet_v3") \
    .trigger(processingTime="30 seconds") \
    .start()

print("Query 7c started: saving district severity summary to Parquet.")

Query 7c started: saving district severity summary to Parquet.


In [21]:
# Check streaming query status
queries = {
    "7a": query_7a,
    "7b": query_7b,
    "7c": query_7c
}

for name, q in queries.items():
    print("=" * 80)
    print("Query:", name)
    print("Is active:", q.isActive)
    print("Status:", q.status)
    print("Exception:", q.exception())
    print("Last progress:", q.lastProgress)

Query: 7a
Is active: True
Status: {'message': 'Waiting for next trigger', 'isDataAvailable': True, 'isTriggerActive': False}
Exception: None
Last progress: {
    "id": "3066a448-035e-4f3c-b18a-01345341c022",
    "runId": "767e7afd-66e1-417c-a9c9-2f361a829ea6",
    "name": null,
    "timestamp": "2026-05-28T04:01:00.001Z",
    "batchId": 26,
    "batchDuration": 1729,
    "numInputRows": 5,
    "inputRowsPerSecond": 1.0,
    "processedRowsPerSecond": 2.9,
    "durationMs": {
        "addBatch": 1538,
        "commitOffsets": 18,
        "getBatch": 0,
        "latestOffset": 24,
        "queryPlanning": 89,
        "triggerExecution": 1728,
        "walCommit": 52
    },
    "eventTime": {
        "avg": "2026-05-28T04:00:57.163Z",
        "max": "2026-05-28T04:00:59.000Z",
        "min": "2026-05-28T04:00:55.000Z",
        "watermark": "2026-05-28T04:00:24.000Z"
    },
    "stateOperators": [],
    "sources": [
        {
            "description": "KafkaV2[Subscribe[accident_stream]]",

### Discussion

The status check confirms that the Task 2.7 streaming queries were active and running without exceptions. The progress output shows that Spark was waiting for the next trigger and had successfully processed recent micro-batches. For example, the displayed query shows `isDataAvailable: True`, `isTriggerActive: False`, and `Exception: None`, indicating that the stream was stable and no runtime error occurred during the Parquet writing process.


In [22]:
# Check Parquet output folders
import os

parquet_paths = [
    "A2B/parquet/high_severity_accidents_v3",
    "A2B/parquet/severity_counts_v3",
    "A2B/parquet/district_severity_summary_v3"
]

for path in parquet_paths:
    print("=" * 80)
    print(path)

    if os.path.exists(path):
        print(os.listdir(path)[:10])
    else:
        print("Folder not created yet")

A2B/parquet/high_severity_accidents_v3
['.part-00000-ecebd2b8-189e-4c74-abe5-2ba769800023-c000.snappy.parquet.crc', 'part-00000-64d2b0b9-38aa-4534-9d17-64ed6a7152fa-c000.snappy.parquet', 'part-00000-5b177c84-1d30-476c-853a-f442658dfa44-c000.snappy.parquet', '.part-00000-a3d69b16-4f13-4dc8-b96e-7c7ec382ee2d-c000.snappy.parquet.crc', '.part-00000-e23913d7-3c40-4c75-9ce6-da6abc944fdc-c000.snappy.parquet.crc', 'part-00000-a3d69b16-4f13-4dc8-b96e-7c7ec382ee2d-c000.snappy.parquet', '.part-00000-d0fcf2a7-71dc-4e3a-9f2e-c17098ba82a1-c000.snappy.parquet.crc', 'part-00000-27c64b68-6bf7-4d1b-a342-3712ea116e9c-c000.snappy.parquet', '.part-00000-4a3cbf2a-9dc4-4fae-8e68-163c19af8c45-c000.snappy.parquet.crc', '.part-00000-3d561f32-73ff-47af-b816-0f05f3ceda2f-c000.snappy.parquet.crc']
A2B/parquet/severity_counts_v3
['.part-00003-64633bd8-2bd5-4e81-bb2b-51028b96f153-c000.snappy.parquet.crc', '.part-00000-3b21986d-cdfb-48e9-a498-8069372533fa-c000.snappy.parquet.crc', 'part-00003-cb698978-0649-4998-9a9b-

In [23]:
# Read saved high-severity accidents Parquet output

high_severity_saved_df = spark.read.parquet("A2B/parquet/high_severity_accidents_v3")

high_severity_saved_df.show(20, truncate=False)

print(
    "Number of high-severity records saved:",
    high_severity_saved_df.count()
)

+---------------+-----------------+----+-----------+-------------+------------+---------------------+----------------------+------------------+
|collision_index|accident_event_ts|area|speed_limit|vehicle_count|Peak_Traffic|High_Risk_Environment|predicted_severity_raw|predicted_severity|
+---------------+-----------------+----+-----------+-------------+------------+---------------------+----------------------+------------------+
+---------------+-----------------+----+-----------+-------------+------------+---------------------+----------------------+------------------+

Number of high-severity records saved: 0


In [24]:
# Read saved severity counts Parquet output
spark.read.parquet("A2B/parquet/severity_counts_v3").show(20, truncate=False)

+-------------------+-------------------+------------------+---------------+
|window_start       |window_end         |predicted_severity|total_accidents|
+-------------------+-------------------+------------------+---------------+
|2026-05-28 05:00:50|2026-05-28 05:01:00|4                 |299            |
|2026-05-28 05:00:50|2026-05-28 05:01:00|3                 |252            |
|2026-05-28 05:00:50|2026-05-28 05:01:00|6                 |27             |
|2026-05-28 04:59:00|2026-05-28 04:59:10|3                 |50             |
|2026-05-28 04:59:00|2026-05-28 04:59:10|6                 |2              |
|2026-05-28 04:59:00|2026-05-28 04:59:10|5                 |12             |
|2026-05-28 04:59:10|2026-05-28 04:59:20|2                 |111            |
|2026-05-28 04:59:10|2026-05-28 04:59:20|6                 |26             |
|2026-05-28 04:59:10|2026-05-28 04:59:20|5                 |40             |
|2026-05-28 05:00:00|2026-05-28 05:00:10|4                 |295            |

In [25]:
# Read saved district severity summary Parquet output
spark.read.parquet("A2B/parquet/district_severity_summary_v3").show(20, truncate=False)

+-------------------+-------------------+------------------+------------------+---------------------+-------------------+---------------+
|window_start       |window_end         |local_district    |low_severity_count|medium_severity_count|high_severity_count|total_accidents|
+-------------------+-------------------+------------------+------------------+---------------------+-------------------+---------------+
|2026-05-28 04:59:30|2026-05-28 05:00:00|Thames Valley     |21                |16                   |0                  |37             |
|2026-05-28 04:59:30|2026-05-28 05:00:00|Cheshire          |17                |8                    |0                  |25             |
|2026-05-28 04:59:30|2026-05-28 05:00:00|West Yorkshire    |35                |26                   |0                  |61             |
|2026-05-28 04:59:30|2026-05-28 05:00:00|North Yorkshire   |8                 |10                   |0                  |18             |
|2026-05-28 04:59:30|2026-05-28 05

### Discussion

The Task 2.7 outputs were successfully written to Parquet files and read back for verification. The high-severity Parquet output contains 0 records, which matches the earlier result where no accidents satisfied the `predicted_severity > 7` condition. The severity counts and district severity summary outputs contain valid window-based aggregation results, confirming that the 10-second severity counts and 30-second district summaries were saved correctly for later Kafka publishing in Task 2.8.


8. Read the Parquet files from task 7 as data streams and send them to Kafka topics with appropriate names.  
(Note: You shall read the parquet files as a streaming data frame and send messages to the Kafka topic when new data appears in the parquet file.)

### Instructions for Rerunning Task 2.8

Task 2.8 reads the Parquet outputs from Task 2.7 and publishes them to Kafka topics. If this notebook is rerun from a clean state or if the existing Kafka topics/checkpoints have already been used, the Kafka topic names and checkpoint folder names should be updated to a new version number, such as changing `_v4` to `_v5`.

For example:

- `severity_counts_topic_v4` → `severity_counts_topic_v5`
- `district_severity_summary_topic_v4` → `district_severity_summary_topic_v5`
- `query_8_stream2_severity_counts_to_kafka_v4` → `query_8_stream2_severity_counts_to_kafka_v5`

The Parquet input paths can stay the same if Task 2.7 still writes to the same Parquet folders. Only the Task 2.8 Kafka topic names, checkpoint names, and verification `topics_to_check` list need to match the chosen version.

In [26]:
for q in spark.streams.active:
    q.stop()

print("Active streaming queries:", len(spark.streams.active))

Active streaming queries: 0


In [27]:
from pyspark.sql.functions import col, to_json, struct, concat_ws

kafka_bootstrap_servers = "kafka:9092"

In [28]:
# Stream 1
# Read high-severity Parquet output from Task 2.7 and publish it to Kafka

high_severity_parquet_path = "A2B/parquet/high_severity_accidents_v3"

# Read the schema from the existing Parquet output
# Spark streaming requires the schema to be provided when reading Parquet as a stream

high_severity_schema = spark.read.parquet(high_severity_parquet_path).schema

# Read the high-severity Parquet folder as a streaming DataFrame
# New Parquet files added to this folder will be processed as streaming input

high_severity_parquet_stream_df = spark.readStream \
    .schema(high_severity_schema) \
    .parquet(high_severity_parquet_path)

# Convert each row into Kafka key-value format
# The collision_index is used as the Kafka key
# The full row is converted to a JSON string as the Kafka value

high_severity_kafka_df = high_severity_parquet_stream_df.select(
    col("collision_index").cast("string").alias("key"),
    to_json(
        struct(*[col(c) for c in high_severity_parquet_stream_df.columns])
    ).alias("value")
)

# Write the high-severity stream to a Kafka topic
# Use a unique checkpoint folder when rerunning this stream

query_8_stream1 = high_severity_kafka_df.writeStream \
    .outputMode("append") \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_bootstrap_servers) \
    .option("topic", "high_severity_accidents_topic_v4") \
    .option("checkpointLocation", "A2B/checkpoint/query_8_stream1_high_severity_to_kafka_v4") \
    .trigger(processingTime="5 seconds") \
    .start()

print("Task 8 Stream 1 started.")

Task 8 Stream 1 started.


In [29]:
# Stream 2
# Read severity counts Parquet output from Task 2.7 and publish it to Kafka

severity_counts_parquet_path = "A2B/parquet/severity_counts_v3"

# Read schema from the existing Parquet output for streaming input

severity_counts_schema = spark.read.parquet(severity_counts_parquet_path).schema

# Read the severity counts Parquet folder as a streaming DataFrame

severity_counts_parquet_stream_df = spark.readStream \
    .schema(severity_counts_schema) \
    .parquet(severity_counts_parquet_path)

# Convert each row into Kafka key-value format
# The key combines window_start and predicted_severity
# The full row is converted to JSON as the Kafka value

severity_counts_kafka_df = severity_counts_parquet_stream_df.select(
    concat_ws(
        "_",
        col("window_start").cast("string"),
        col("predicted_severity").cast("string")
    ).alias("key"),
    to_json(
        struct(*[col(c) for c in severity_counts_parquet_stream_df.columns])
    ).alias("value")
)

# Write the severity counts stream to Kafka
# Use a unique checkpoint folder when rerunning the stream

query_8_stream2 = severity_counts_kafka_df.writeStream \
    .outputMode("append") \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_bootstrap_servers) \
    .option("topic", "severity_counts_topic_v4") \
    .option("checkpointLocation", "A2B/checkpoint/query_8_stream2_severity_counts_to_kafka_v4") \
    .trigger(processingTime="10 seconds") \
    .start()

print("Task 8 Stream 2 started.")

Task 8 Stream 2 started.


In [30]:
# Stream 3
# Read district severity summary Parquet output from Task 2.7 and publish it to Kafka

district_summary_parquet_path = "A2B/parquet/district_severity_summary_v3"

# Read schema from the existing Parquet output for streaming input

district_summary_schema = spark.read.parquet(district_summary_parquet_path).schema

# Read the district summary Parquet folder as a streaming DataFrame

district_summary_parquet_stream_df = spark.readStream \
    .schema(district_summary_schema) \
    .parquet(district_summary_parquet_path)

# Convert each row into Kafka key-value format
# The key combines window_start and local_district
# The full row is converted to JSON as the Kafka value

district_summary_kafka_df = district_summary_parquet_stream_df.select(
    concat_ws(
        "_",
        col("window_start").cast("string"),
        col("local_district").cast("string")
    ).alias("key"),
    to_json(
        struct(*[col(c) for c in district_summary_parquet_stream_df.columns])
    ).alias("value")
)

# Write the district summary stream to Kafka
# Use a unique checkpoint folder when rerunning the stream

query_8_stream3 = district_summary_kafka_df.writeStream \
    .outputMode("append") \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_bootstrap_servers) \
    .option("topic", "district_severity_summary_topic_v4") \
    .option("checkpointLocation", "A2B/checkpoint/query_8_stream3_district_summary_to_kafka_v4") \
    .trigger(processingTime="30 seconds") \
    .start()

print("Task 8 Stream 3 started.")

Task 8 Stream 3 started.


In [31]:
# Check the status and progress of the three Task 2.8 Kafka
# writing streams
# Each query sends one Parquet stream to a Kafka topic:
# Stream 1: high-severity accident records
# Stream 2: severity count summary
# Stream 3: district severity summary
# Store the three streaming query objects in a dictionary
# This makes it easier to check their status using a loop

queries_8 = {
    "Stream 1 - high severity": query_8_stream1,
    "Stream 2 - severity counts": query_8_stream2,
    "Stream 3 - district summary": query_8_stream3
}

# Print the running status and recent progress for each stream
for name, q in queries_8.items():
    print("=" * 100)
    print(name)
    print("Is active:", q.isActive) # Check whether the stream is still active
    print("Exception:", q.exception()) # Print any exception if the stream has failed
    
    # recentProgress stores recent micro-batch information
    # numInputRows shows how many rows Spark read in each batch
    # numOutputRows shows how many rows were written to Kafka

    for p in q.recentProgress:
        print(
            "batchId:", p.get("batchId"),
            "| numInputRows:", p.get("numInputRows"),
            "| numOutputRows:", p.get("sink", {}).get("numOutputRows")
        )

Stream 1 - high severity
Is active: True
Exception: None
batchId: 0 | numInputRows: 0 | numOutputRows: 0
batchId: 1 | numInputRows: 0 | numOutputRows: 0
batchId: 1 | numInputRows: 0 | numOutputRows: 0
batchId: 1 | numInputRows: 0 | numOutputRows: 0
batchId: 1 | numInputRows: 0 | numOutputRows: 0
batchId: 1 | numInputRows: 0 | numOutputRows: 0
batchId: 1 | numInputRows: 0 | numOutputRows: 0
batchId: 1 | numInputRows: 0 | numOutputRows: 0
batchId: 1 | numInputRows: 0 | numOutputRows: 0
batchId: 1 | numInputRows: 0 | numOutputRows: 0
batchId: 1 | numInputRows: 0 | numOutputRows: 0
batchId: 1 | numInputRows: 0 | numOutputRows: 0
Stream 2 - severity counts
Is active: True
Exception: None
batchId: 0 | numInputRows: 76 | numOutputRows: 76
batchId: 1 | numInputRows: 0 | numOutputRows: 0
batchId: 1 | numInputRows: 0 | numOutputRows: 0
batchId: 1 | numInputRows: 0 | numOutputRows: 0
batchId: 1 | numInputRows: 0 | numOutputRows: 0
batchId: 1 | numInputRows: 0 | numOutputRows: 0
batchId: 1 | numIn

### Discussion

The Task 2.8 streaming queries were checked after reading the saved Parquet outputs as streams and writing them to Kafka topics. All three streams were active and had no exceptions. Stream 1 for high-severity records showed 0 input and output rows because no `predicted_severity > 7` records were saved in the current run. Stream 2 successfully wrote 76 severity count records to Kafka, while Stream 3 successfully wrote 182 district summary records. This confirms that the available Parquet outputs were successfully published to the corresponding Kafka topics.


In [34]:
# Verification step to confirm that the summary streams were successfully written to Kafka

# Kafka topics to verify
topics_to_check = [
    "severity_counts_topic_v4",
    "district_severity_summary_topic_v4"
]

# Read each Kafka topic from the beginning to the latest available offset
# and print the total number of messages
for topic in topics_to_check:
    print("=" * 100)
    print("Topic:", topic)

    kafka_df = spark.read \
        .format("kafka") \
        .option("kafka.bootstrap.servers", kafka_bootstrap_servers) \
        .option("subscribe", topic) \
        .option("startingOffsets", "earliest") \
        .option("endingOffsets", "latest") \
        .load()
    
    # Count all messages currently stored in the topic
    print("Message count:", kafka_df.count())
    
    # Display a small sample of Kafka key-value messages
    # Kafka stores key and value as binary, so they are cast to string for readability
    kafka_df.select(
        col("key").cast("string").alias("key"),
        col("value").cast("string").alias("value")
    ).show(5, truncate=False)

Topic: severity_counts_topic_v4
Message count: 76
+---------------------+------------------------------------------------------------------------------------------------------------------------------------------+
|key                  |value                                                                                                                                     |
+---------------------+------------------------------------------------------------------------------------------------------------------------------------------+
|2026-05-28 05:01:00_2|{"window_start":"2026-05-28T05:01:00.000+01:00","window_end":"2026-05-28T05:01:10.000+01:00","predicted_severity":2,"total_accidents":117}|
|2026-05-28 04:59:20_5|{"window_start":"2026-05-28T04:59:20.000+01:00","window_end":"2026-05-28T04:59:30.000+01:00","predicted_severity":5,"total_accidents":54} |
|2026-05-28 05:00:30_4|{"window_start":"2026-05-28T05:00:30.000+01:00","window_end":"2026-05-28T05:00:40.000+01:00","predicted_severity

### Discussion

The Kafka topics created in Task 2.8 were read back to verify that the Parquet stream outputs were successfully published. The `severity_counts_topic_v4` topic contains 76 messages, and the sample records show JSON values with `window_start`, `window_end`, `predicted_severity`, and `total_accidents`. The `district_severity_summary_topic_v4` topic contains 182 messages, with each message storing district-level low, medium, high, and total accident counts. This confirms that the main streaming summary outputs were successfully sent to Kafka in the expected key-value format.
